# Packages

In [7]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import random as rd
from surprise import AlgoBase
from surprise.prediction_algorithms.predictions import PredictionImpossible
from sklearn.linear_model import LinearRegression


from loaders import load_ratings
from loaders import load_items
from constants import Constant as C

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Explore and select content features

In [2]:
df_items = load_items()
df_ratings = load_ratings()

# Example 1 : create title_length features
df_features = df_items[C.LABEL_COL].apply(lambda x: len(x)).to_frame('n_character_title')
display(df_features.head())

# (explore here other features)


,n_character_title
movieId,
4993,57
5952,45
527,23
2028,26
4308,19


# Build a content-based model
When ready, move the following class in the *models.py* script

In [ ]:
class ContentBased(AlgoBase):
    def __init__(self, features_method, regressor_method):
        AlgoBase.__init__(self)
        self.regressor_method = regressor_method
        self.content_features = self.create_content_features(features_method)

    def create_content_features(self, features_method):
        """Content Analyzer"""
        df_items = load_items()
        if features_method is None:
            df_features = None
        elif features_method == "title_length": # a naive method that creates only 1 feature based on title length
            df_features = df_items[C.LABEL_COL].apply(lambda x: len(x)).to_frame('n_character_title')
        else: # (implement other feature creations here)
            raise NotImplementedError(f'Feature method {features_method} not yet implemented')
        return df_features
    

    def fit(self, trainset):
        """Profile Learner"""
        AlgoBase.fit(self, trainset)
        
        # Preallocate user profiles
        self.user_profile = {u: None for u in trainset.all_users()}

        if self.regressor_method == 'random_score':
            pass
        
        elif self.regressor_method == 'random_sample':
            for u in self.user_profile:
                self.user_profile[u] = [rating for _, rating in self.trainset.ur[u]]
        else:
            pass
            # (implement here the regressor fitting)  
        
    def estimate(self, u, i):
        """Scoring component used for item filtering"""
        # First, handle cases for unknown users and items
        if not (self.trainset.knows_user(u) and self.trainset.knows_item(i)):
            raise PredictionImpossible('User and/or item is unkown.')


        if self.regressor_method == 'random_score':
            rd.seed()
            score = rd.uniform(0.5,5)

        elif self.regressor_method == 'random_sample':
            rd.seed()
            score = rd.choice(self.user_profile[u])
        
        # (implement here the regressor prediction)
        # Linear regression 
        elif self.regressor_method in (
            'linear_regression', 'random_forest', 'ridge', 'ridge_cv', 'ridge_cv_bias'
        ):
            feature_names = list(self.content_features.columns)
            for u in self.user_profile:
                df_user = pd.DataFrame(self.trainset.ur[u], columns=['item_id', 'user_ratings'])
                df_user['item_id'] = df_user['item_id'].map(self.trainset.to_raw_iid)

                df_user = df_user.merge(
                    self.content_features,
                    how='left',
                    left_on='item_id',
                    right_index=True
                )

                # Remove items without any feature
                df_user = df_user.dropna(subset=feature_names, how='all')
                df_user = df_user.fillna(0)

                # Guard : not enough examples to fit
                if len(df_user) < 2:
                    self.user_profile[u] = None
                    continue

                X = df_user[feature_names].values
                y = df_user['user_ratings'].values

                if self.regressor_method == 'linear_regression':
                    regressor = LinearRegression(fit_intercept=True)
            
                regressor.fit(X, y)
                self.user_profile[u] = regressor


        return score


The following script test the ContentBased class

In [6]:
def test_contentbased_class(feature_method, regressor_method):
    """Test the ContentBased class.
    Tries to make a prediction on the first (user,item ) tuple of the anti_test_set
    """
    sp_ratings = load_ratings(surprise_format=True)
    train_set = sp_ratings.build_full_trainset()
    content_algo = ContentBased(feature_method, regressor_method)
    content_algo.fit(train_set)
    anti_test_set_first = train_set.build_anti_testset()[0]
    prediction = content_algo.predict(anti_test_set_first[0], anti_test_set_first[1])
    print(prediction)

# (call here the test functions with different regressor methods)

# Test 1 : random score (no features)
print("=== Test with random_score ===")
test_contentbased_class(None, 'random_score')

# Test 2 : random sample (no features)
print("=== Test with random_sample ===")
test_contentbased_class(None, 'random_sample')


=== Test with random_score ===
user: 11         item: 1214       r_ui = None   est = 4.77   {'was_impossible': False}
=== Test with random_sample ===
user: 11         item: 1214       r_ui = None   est = 4.50   {'was_impossible': False}
